In [1]:
# Notebook 5: IDRiD Cross-Dataset Evaluation
# Project: Diabetic Retinopathy Screening

from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import models, transforms

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Notebook 5: IDRiD Evaluation")
print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Notebook 5: IDRiD Evaluation
Device: cuda
PyTorch version: 2.11.0+cu128
CUDA available: True


In [2]:
from pathlib import Path
import pandas as pd

# IDRiD paths
IDRID_ROOT = Path("/kaggle/input/datasets/mariaherrerot/idrid-dataset")
IDRID_IMAGE_DIR = IDRID_ROOT / "Imagenes" / "Imagenes"
IDRID_LABELS_PATH = IDRID_ROOT / "idrid_labels.csv"

# Verify paths
print("Image directory exists:", IDRID_IMAGE_DIR.exists())
print("Labels file exists:", IDRID_LABELS_PATH.exists())

# Load labels
idrid_df = pd.read_csv(IDRID_LABELS_PATH)

print("\nLabel CSV shape:", idrid_df.shape)
print("\nColumn names:")
print(idrid_df.columns.tolist())

print("\nFirst 5 label rows:")
display(idrid_df.head())

# Inspect image files
image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
image_paths = sorted(
    p for p in IDRID_IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

print("\nTotal images found:", len(image_paths))
print("\nFirst 10 image filenames:")
for path in image_paths[:10]:
    print(path.name)

Image directory exists: True
Labels file exists: True

Label CSV shape: (455, 12)

Column names:
['id_code', 'diagnosis', 'Risk of macular edema ', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

First 5 label rows:


,id_code,diagnosis,Risk of macular edema,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,IDRiD_001,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IDRiD_002,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IDRiD_003,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IDRiD_004,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IDRiD_005,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Total images found: 455

First 10 image filenames:
IDRiD_001.jpg
IDRiD_001test.jpg
IDRiD_002.jpg
IDRiD_003.jpg
IDRiD_004.jpg
IDRiD_004test.jpg
IDRiD_005.jpg
IDRiD_005test.jpg
IDRiD_006.jpg
IDRiD_006test.jpg


In [3]:
# Remove empty CSV columns
idrid_df = idrid_df.dropna(axis=1, how="all").copy()

# Normalize column names
idrid_df.columns = idrid_df.columns.str.strip()

# Inspect diagnosis labels
print("Diagnosis class distribution:")
print(idrid_df["diagnosis"].value_counts().sort_index())

print("\nUnique diagnosis values:")
print(sorted(idrid_df["diagnosis"].dropna().unique()))

# Check duplicate label IDs
print("\nDuplicate label IDs:", idrid_df["id_code"].duplicated().sum())

# Check exact image matches
image_name_set = {p.name for p in image_paths}

idrid_df["expected_filename"] = idrid_df["id_code"] + ".jpg"
idrid_df["image_exists"] = idrid_df["expected_filename"].isin(image_name_set)

print("\nExact filename matches:")
print(idrid_df["image_exists"].value_counts())

print("\nLabel IDs without an exact matching image:")
display(
    idrid_df.loc[
        ~idrid_df["image_exists"],
        ["id_code", "expected_filename", "diagnosis"]
    ].head(20)
)

# Inspect filenames containing 'test'
test_named_images = [p.name for p in image_paths if "test" in p.stem.lower()]

print("\nImages containing 'test' in filename:", len(test_named_images))
print("Examples:", test_named_images[:15])

Diagnosis class distribution:
diagnosis
0    129
1     22
2    156
3     84
4     64
Name: count, dtype: int64

Unique diagnosis values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Duplicate label IDs: 0

Exact filename matches:
image_exists
True    455
Name: count, dtype: int64

Label IDs without an exact matching image:


,id_code,expected_filename,diagnosis



Images containing 'test' in filename: 80
Examples: ['IDRiD_001test.jpg', 'IDRiD_004test.jpg', 'IDRiD_005test.jpg', 'IDRiD_006test.jpg', 'IDRiD_007test.jpg', 'IDRiD_008test.jpg', 'IDRiD_009test.jpg', 'IDRiD_010test.jpg', 'IDRiD_011test.jpg', 'IDRiD_012test.jpg', 'IDRiD_013test.jpg', 'IDRiD_014test.jpg', 'IDRiD_015test.jpg', 'IDRiD_016test.jpg', 'IDRiD_017test.jpg']


In [4]:
from pathlib import Path
import tensorflow as tf

# Saved model paths
RESNET_PATH = Path(
    "/kaggle/input/models/mjayanth007/resnet/keras/default/1/"
    "resnet_outputs/resnet50_best.keras"
)

EFFICIENTNET_PATH = Path(
    "/kaggle/input/models/mjayanth007/efficientnetfinetuned/keras/default/1/"
    "efficientnet_outputs/efficientnetb0_finetuned_best.keras"
)

# Verify model files
assert RESNET_PATH.exists(), f"ResNet model not found: {RESNET_PATH}"
assert EFFICIENTNET_PATH.exists(), (
    f"EfficientNet model not found: {EFFICIENTNET_PATH}"
)

print("ResNet file size:",
      f"{RESNET_PATH.stat().st_size / (1024**2):.2f} MB")
print("EfficientNet file size:",
      f"{EFFICIENTNET_PATH.stat().st_size / (1024**2):.2f} MB")

# Load trained models
resnet_model = tf.keras.models.load_model(
    RESNET_PATH,
    compile=False
)

efficientnet_model = tf.keras.models.load_model(
    EFFICIENTNET_PATH,
    compile=False
)

print("\nBoth models loaded successfully.")

print("\nResNet input shape:", resnet_model.input_shape)
print("ResNet output shape:", resnet_model.output_shape)

print("\nEfficientNet input shape:", efficientnet_model.input_shape)
print("EfficientNet output shape:", efficientnet_model.output_shape)

ResNet file size: 90.73 MB
EfficientNet file size: 27.65 MB


I0000 00:00:1792133301.866221      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792133301.870178      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



Both models loaded successfully.

ResNet input shape: (None, 224, 224, 3)
ResNet output shape: (None, 5)

EfficientNet input shape: (None, 224, 224, 3)
EfficientNet output shape: (None, 5)


In [5]:
# Inspect model architecture and preprocessing-related layers

def inspect_model(model, model_name):
    print(f"\n{'=' * 60}")
    print(model_name)
    print("=" * 60)

    print("Input shape:", model.input_shape)
    print("Output shape:", model.output_shape)

    print("\nFirst 15 layers:")
    for layer in model.layers[:15]:
        print(
            f"{layer.name:35s} "
            f"{layer.__class__.__name__:25s} "
            f"output={getattr(layer, 'output_shape', 'see model summary')}"
        )

    print("\nModel summary:")
    model.summary()

inspect_model(resnet_model, "ResNet50")
inspect_model(efficientnet_model, "EfficientNetB0")


ResNet50
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)

First 15 layers:
input_layer_1                       InputLayer                output=see model summary
resnet50                            Functional                output=(None, 7, 7, 2048)
global_average_pooling2d            GlobalAveragePooling2D    output=see model summary
dropout                             Dropout                   output=see model summary
dense                               Dense                     output=see model summary

Model summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

... (2 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 1,489,765 (5.68 MB)

 Non-trainable params: 2,566,211 (9.79 MB)

In [6]:
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

IMAGE_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Build evaluation dataframe from the verified IDRiD labels
idrid_eval_df = idrid_df[["id_code", "diagnosis"]].copy()

idrid_eval_df["image_path"] = idrid_eval_df["id_code"].apply(
    lambda image_id: str(IDRID_IMAGE_DIR / f"{image_id}.jpg")
)

# Verify image paths and labels
idrid_eval_df["file_exists"] = idrid_eval_df["image_path"].apply(
    lambda path: Path(path).is_file()
)

assert idrid_eval_df["file_exists"].all(), "Missing labeled images."
assert idrid_eval_df["id_code"].is_unique, "Duplicate image IDs found."
assert idrid_eval_df["diagnosis"].between(0, 4).all(), (
    "Unexpected diagnosis labels found."
)

idrid_eval_df = idrid_eval_df.drop(columns="file_exists")

def load_idrid_image(image_path, label, model_type):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    if model_type == "resnet":
        image = tf.keras.applications.resnet50.preprocess_input(image)
    else:
        # Match EfficientNet training: resized RGB values,
        # with preprocessing handled inside the Keras model.
        image = tf.cast(image, tf.float32)

    return image, tf.cast(label, tf.int32)

def create_idrid_eval_dataset(dataframe, model_type):
    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe["image_path"].to_numpy(),
        dataframe["diagnosis"].to_numpy(dtype=np.int32)
    ))

    dataset = dataset.map(
        lambda path, label: load_idrid_image(
            path, label, model_type
        ),
        num_parallel_calls=AUTOTUNE
    )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

resnet_eval_ds = create_idrid_eval_dataset(
    idrid_eval_df, "resnet"
)

efficientnet_eval_ds = create_idrid_eval_dataset(
    idrid_eval_df, "efficientnet"
)

print("IDRiD evaluation dataset prepared.")
print("Total labeled images:", len(idrid_eval_df))
print("\nDiagnosis distribution:")
print(idrid_eval_df["diagnosis"].value_counts().sort_index())

for images, labels in resnet_eval_ds.take(1):
    print("\nResNet batch shape:", images.shape)
    print("ResNet pixel range:",
          float(tf.reduce_min(images)),
          "to", float(tf.reduce_max(images)))

for images, labels in efficientnet_eval_ds.take(1):
    print("\nEfficientNet batch shape:", images.shape)
    print("EfficientNet pixel range:",
          float(tf.reduce_min(images)),
          "to", float(tf.reduce_max(images)))

IDRiD evaluation dataset prepared.
Total labeled images: 455

Diagnosis distribution:
diagnosis
0    129
1     22
2    156
3     84
4     64
Name: count, dtype: int64

ResNet batch shape: (32, 224, 224, 3)
ResNet pixel range: -123.68000030517578 to 138.22100830078125

EfficientNet batch shape: (32, 224, 224, 3)
EfficientNet pixel range: 0.0 to 255.0


In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

def evaluate_model(model, dataset, model_name):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        probabilities = model(images, training=False).numpy()
        predictions = np.argmax(probabilities, axis=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.tolist())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    print(f"\n{'=' * 60}")
    print(f"{model_name} — IDRiD Evaluation")
    print("=" * 60)

    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall (macro):    {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1-score (macro):  {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1-score (weighted): {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=[
            "No DR",
            "Mild",
            "Moderate",
            "Severe",
            "Proliferative DR"
        ],
        digits=4,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])

    print("Confusion matrix (rows=true, columns=predicted):")
    print(cm)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": cm,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }


resnet_results = evaluate_model(
    resnet_model,
    resnet_eval_ds,
    "ResNet50"
)

efficientnet_results = evaluate_model(
    efficientnet_model,
    efficientnet_eval_ds,
    "EfficientNetB0"
)


ResNet50 — IDRiD Evaluation
Accuracy:  0.3890
Precision (macro): 0.3696
Recall (macro):    0.2977
F1-score (macro):  0.2409
F1-score (weighted): 0.2972

Classification report:
                  precision    recall  f1-score   support

           No DR     0.3707    1.0000    0.5409       129
            Mild     0.0000    0.0000    0.0000        22
        Moderate     0.3836    0.1795    0.2445       156
          Severe     0.5000    0.0119    0.0233        84
Proliferative DR     0.5938    0.2969    0.3958        64

        accuracy                         0.3890       455
       macro avg     0.3696    0.2977    0.2409       455
    weighted avg     0.4124    0.3890    0.2972       455

Confusion matrix (rows=true, columns=predicted):
[[129   0   0   0   0]
 [ 22   0   0   0   0]
 [124   0  28   0   4]
 [ 50   0  24   1   9]
 [ 23   0  21   1  19]]

EfficientNetB0 — IDRiD Evaluation
Accuracy:  0.3033
Precision (macro): 0.1153
Recall (macro):    0.2113
F1-score (macro):  0.1187
F1

In [8]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Match the final preprocessing function from Notebook 2
RGB_MEAN = np.array(
    [133.71985309, 79.52423862, 41.30843877],
    dtype=np.float32
)

RGB_STD = np.array(
    [62.56835813, 38.40553729, 29.33583564],
    dtype=np.float32
)

IDRID_PROCESSED_DIR = Path(
    "/kaggle/working/idrid_preprocessed"
)
IDRID_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def resize_with_padding(image, target_size=512):
    height, width = image.shape[:2]

    scale = target_size / max(height, width)
    new_width = round(width * scale)
    new_height = round(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    canvas = np.zeros(
        (target_size, target_size, 3),
        dtype=image.dtype
    )

    top = (target_size - new_height) // 2
    left = (target_size - new_width) // 2

    canvas[
        top:top + new_height,
        left:left + new_width
    ] = resized

    return canvas


def preprocess_fundus(image_path, target_size=512):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Unable to read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # CLAHE on the LAB lightness channel
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=5.0,
        tileGridSize=(8, 8)
    )
    enhanced_l = clahe.apply(l_channel)

    enhanced_lab = cv2.merge(
        [enhanced_l, a_channel, b_channel]
    )
    enhanced_rgb = cv2.cvtColor(
        enhanced_lab, cv2.COLOR_LAB2RGB
    )

    # Gaussian filtering
    filtered = cv2.GaussianBlur(
        enhanced_rgb, (5, 5), sigmaX=0
    )

    # Crop dark borders
    gray = cv2.cvtColor(filtered, cv2.COLOR_RGB2GRAY)
    mask = gray > 40

    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]

    if len(rows) == 0 or len(cols) == 0:
        raise ValueError(
            f"No foreground detected: {image_path}"
        )

    cropped = filtered[
        rows[0]:rows[-1] + 1,
        cols[0]:cols[-1] + 1
    ]

    # Resize to 512 x 512 with aspect ratio preserved
    resized = resize_with_padding(cropped, target_size)

    # Match Notebook 2 normalization
    normalized = (
        resized.astype(np.float32) - RGB_MEAN
    ) / RGB_STD

    return normalized


# Use only images matched to the 455 labeled CSV entries
idrid_processed_df = idrid_df[
    ["id_code", "diagnosis"]
].copy()

idrid_processed_df["image_path"] = idrid_processed_df[
    "id_code"
].apply(
    lambda image_id: str(
        IDRID_PROCESSED_DIR / f"{image_id}.jpg"
    )
)

for row in tqdm(
    idrid_processed_df.itertuples(index=False),
    total=len(idrid_processed_df),
    desc="Preprocessing IDRiD"
):
    source_path = IDRID_IMAGE_DIR / f"{row.id_code}.jpg"

    processed = preprocess_fundus(source_path)

    # Match Notebook 2: restore RGB pixel values before JPEG saving
    image_uint8 = np.clip(
        processed * RGB_STD + RGB_MEAN,
        0, 255
    ).astype(np.uint8)

    output_path = IDRID_PROCESSED_DIR / f"{row.id_code}.jpg"

    success = cv2.imwrite(
        str(output_path),
        cv2.cvtColor(image_uint8, cv2.COLOR_RGB2BGR),
        [cv2.IMWRITE_JPEG_QUALITY, 95]
    )

    if not success:
        raise IOError(f"Failed to save {output_path}")

assert all(
    Path(p).is_file()
    for p in idrid_processed_df["image_path"]
)

print("\nIDRiD preprocessing complete.")
print("Processed images:", len(idrid_processed_df))
print("Output directory:", IDRID_PROCESSED_DIR)
print(
    "Sample image shape:",
    cv2.imread(idrid_processed_df.iloc[0]["image_path"]).shape
)
print("\nDiagnosis distribution:")
print(idrid_processed_df["diagnosis"].value_counts().sort_index())

Preprocessing IDRiD:   0%|          | 0/455 [00:00<?, ?it/s]


IDRiD preprocessing complete.
Processed images: 455
Output directory: /kaggle/working/idrid_preprocessed
Sample image shape: (512, 512, 3)

Diagnosis distribution:
diagnosis
0    129
1     22
2    156
3     84
4     64
Name: count, dtype: int64


In [9]:
import tensorflow as tf
import numpy as np
from pathlib import Path

IMAGE_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_processed_image(image_path, label, model_type):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)

    # Resize preprocessed 512x512 image to model input size
    image = tf.image.resize(
        image, [IMAGE_SIZE, IMAGE_SIZE]
    )

    # Match each model's training-time input transformation
    if model_type == "resnet":
        image = tf.keras.applications.resnet50.preprocess_input(
            image
        )
    else:
        image = tf.cast(image, tf.float32)

    return image, tf.cast(label, tf.int32)


def create_processed_eval_dataset(dataframe, model_type):
    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe["image_path"].to_numpy(),
        dataframe["diagnosis"].to_numpy(dtype=np.int32)
    ))

    dataset = dataset.map(
        lambda path, label: load_processed_image(
            path, label, model_type
        ),
        num_parallel_calls=AUTOTUNE
    )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


resnet_processed_eval_ds = create_processed_eval_dataset(
    idrid_processed_df, "resnet"
)

efficientnet_processed_eval_ds = create_processed_eval_dataset(
    idrid_processed_df, "efficientnet"
)

print("Processed-image evaluation datasets created.")

for images, labels in resnet_processed_eval_ds.take(1):
    print("ResNet batch shape:", images.shape)
    print(
        "ResNet pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )

for images, labels in efficientnet_processed_eval_ds.take(1):
    print("EfficientNet batch shape:", images.shape)
    print(
        "EfficientNet pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )

Processed-image evaluation datasets created.
ResNet batch shape: (32, 224, 224, 3)
ResNet pixel range: -123.68000030517578 to 138.22100830078125
EfficientNet batch shape: (32, 224, 224, 3)
EfficientNet pixel range: 0.0 to 255.0


In [10]:
resnet_processed_results = evaluate_model(
    resnet_model,
    resnet_processed_eval_ds,
    "ResNet50 — Preprocessed IDRiD"
)

efficientnet_processed_results = evaluate_model(
    efficientnet_model,
    efficientnet_processed_eval_ds,
    "EfficientNetB0 — Preprocessed IDRiD"
)


ResNet50 — Preprocessed IDRiD — IDRiD Evaluation
Accuracy:  0.5473
Precision (macro): 0.4873
Recall (macro):    0.4847
F1-score (macro):  0.4656
F1-score (weighted): 0.5446

Classification report:
                  precision    recall  f1-score   support

           No DR     0.8125    0.7054    0.7552       129
            Mild     0.1887    0.4545    0.2667        22
        Moderate     0.5431    0.6859    0.6062       156
          Severe     0.3469    0.2024    0.2556        84
Proliferative DR     0.5455    0.3750    0.4444        64

        accuracy                         0.5473       455
       macro avg     0.4873    0.4847    0.4656       455
    weighted avg     0.5665    0.5473    0.5446       455

Confusion matrix (rows=true, columns=predicted):
[[ 91  24  14   0   0]
 [  9  10   3   0   0]
 [ 11  19 107  10   9]
 [  0   0  56  17  11]
 [  1   0  17  22  24]]

EfficientNetB0 — Preprocessed IDRiD — IDRiD Evaluation
Accuracy:  0.5253
Precision (macro): 0.4889
Recall (macr

In [11]:
import tensorflow as tf
from pathlib import Path

RESNET_FINETUNED_PATH = Path(
    "/kaggle/input/models/mjayanth007/resnet/keras/default/1/"
    "resnet_outputs/resnet50_finetuned_best.keras"
)

assert RESNET_FINETUNED_PATH.exists(), (
    f"Checkpoint not found: {RESNET_FINETUNED_PATH}"
)

resnet_finetuned_model = tf.keras.models.load_model(
    RESNET_FINETUNED_PATH,
    compile=False
)

print("Fine-tuned ResNet50 loaded successfully.")
print("Input shape:", resnet_finetuned_model.input_shape)
print("Output shape:", resnet_finetuned_model.output_shape)

resnet_finetuned_processed_results = evaluate_model(
    resnet_finetuned_model,
    resnet_processed_eval_ds,
    "ResNet50 Fine-tuned — Preprocessed IDRiD"
)

Fine-tuned ResNet50 loaded successfully.
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)

ResNet50 Fine-tuned — Preprocessed IDRiD — IDRiD Evaluation
Accuracy:  0.5231
Precision (macro): 0.5092
Recall (macro):    0.4837
F1-score (macro):  0.4491
F1-score (weighted): 0.5236

Classification report:
                  precision    recall  f1-score   support

           No DR     0.8276    0.5581    0.6667       129
            Mild     0.1667    0.5909    0.2600        22
        Moderate     0.5421    0.7436    0.6270       156
          Severe     0.3333    0.1667    0.2222        84
Proliferative DR     0.6765    0.3594    0.4694        64

        accuracy                         0.5231       455
       macro avg     0.5092    0.4837    0.4491       455
    weighted avg     0.5852    0.5231    0.5236       455

Confusion matrix (rows=true, columns=predicted):
[[ 72  43  14   0   0]
 [  6  13   3   0   0]
 [  8  22 116   6   4]
 [  0   0  63  14   7]
 [  1   0  18  22  23]]


In [12]:
comparison_df = pd.DataFrame([
    {
        "Model": "ResNet50 best",
        "Accuracy": resnet_processed_results["accuracy"],
        "Macro F1": resnet_processed_results["macro_f1"],
        "Weighted F1": resnet_processed_results["weighted_f1"]
    },
    {
        "Model": "ResNet50 fine-tuned",
        "Accuracy": resnet_finetuned_processed_results["accuracy"],
        "Macro F1": resnet_finetuned_processed_results["macro_f1"],
        "Weighted F1": resnet_finetuned_processed_results["weighted_f1"]
    },
    {
        "Model": "EfficientNetB0 fine-tuned",
        "Accuracy": efficientnet_processed_results["accuracy"],
        "Macro F1": efficientnet_processed_results["macro_f1"],
        "Weighted F1": efficientnet_processed_results["weighted_f1"]
    }
])

comparison_df = comparison_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

display(
    comparison_df.style.format({
        "Accuracy": "{:.4f}",
        "Macro F1": "{:.4f}",
        "Weighted F1": "{:.4f}"
    })
)

,Model,Accuracy,Macro F1,Weighted F1
0,ResNet50 best,0.5473,0.4656,0.5446
1,EfficientNetB0 fine-tuned,0.5253,0.4583,0.5359
2,ResNet50 fine-tuned,0.5231,0.4491,0.5236


In [13]:
from pathlib import Path

RESULTS_DIR = Path("/kaggle/working/idrid_evaluation_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save model comparison
comparison_df.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False
)

# Save detailed classification reports
from sklearn.metrics import classification_report
import json

evaluation_results = {
    "ResNet50_best": resnet_processed_results,
    "ResNet50_finetuned": resnet_finetuned_processed_results,
    "EfficientNetB0_finetuned": efficientnet_processed_results
}

summary = {}

for model_name, result in evaluation_results.items():
    summary[model_name] = {
        "accuracy": float(result["accuracy"]),
        "macro_f1": float(result["macro_f1"]),
        "weighted_f1": float(result["weighted_f1"]),
        "confusion_matrix": result["confusion_matrix"].tolist()
    }

with open(
    RESULTS_DIR / "evaluation_summary.json",
    "w"
) as file:
    json.dump(summary, file, indent=4)

print("Results saved to:", RESULTS_DIR)
print("\nFiles:")
for path in sorted(RESULTS_DIR.iterdir()):
    print(path.name)

Results saved to: /kaggle/working/idrid_evaluation_results

Files:
evaluation_summary.json
model_comparison.csv


In [14]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/final_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Five-class evaluation results already calculated in Notebook 5
comparison_df.to_csv(
    OUTPUT_DIR / "idrid_five_class_model_comparison.csv",
    index=False
)

# Binary results aggregated from the existing five-class predictions
binary_results = pd.DataFrame([
    {
        "Model": "ResNet50 best",
        "TN": 134,
        "FP": 17,
        "FN": 31,
        "TP": 273,
        "Accuracy": 407 / 455 if False else (134 + 273) / 455,
        "Referable_Sensitivity": 273 / 304,
        "Nonreferable_Specificity": 134 / 151,
        "Referable_Precision": 273 / 290,
        "Referable_F1": 2 * 273 / (2 * 273 + 17 + 31),
    },
    {
        "Model": "ResNet50 fine-tuned",
        "TN": 134,
        "FP": 17,
        "FN": 31,
        "TP": 273,
        "Accuracy": (134 + 273) / 455,
        "Referable_Sensitivity": 273 / 304,
        "Nonreferable_Specificity": 134 / 151,
        "Referable_Precision": 273 / 290,
        "Referable_F1": 2 * 273 / (2 * 273 + 17 + 31),
    },
    {
        "Model": "EfficientNetB0 fine-tuned",
        "TN": 147,
        "FP": 4,
        "FN": 45,
        "TP": 259,
        "Accuracy": (147 + 259) / 455,
        "Referable_Sensitivity": 259 / 304,
        "Nonreferable_Specificity": 147 / 151,
        "Referable_Precision": 259 / 263,
        "Referable_F1": 2 * 259 / (2 * 259 + 4 + 45),
    },
])

binary_results.to_csv(
    OUTPUT_DIR / "idrid_binary_screening_results.csv",
    index=False
)

print("Exported evaluation files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

display(binary_results.round(4))

Exported evaluation files:
idrid_binary_screening_results.csv
idrid_five_class_model_comparison.csv


,Model,TN,FP,FN,TP,Accuracy,Referable_Sensitivity,Nonreferable_Specificity,Referable_Precision,Referable_F1
0,ResNet50 best,134,17,31,273,0.8945,0.898,0.8874,0.9414,0.9192
1,ResNet50 fine-tuned,134,17,31,273,0.8945,0.898,0.8874,0.9414,0.9192
2,EfficientNetB0 fine-tuned,147,4,45,259,0.8923,0.852,0.9735,0.9848,0.9136


In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/final_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

binary_confusion_matrices = {
    "ResNet50_best": np.array([
        [134, 17],
        [31, 273]
    ]),
    "ResNet50_finetuned": np.array([
        [134, 17],
        [31, 273]
    ]),
    "EfficientNetB0_finetuned": np.array([
        [147, 4],
        [45, 259]
    ])
}

for model_name, cm in binary_confusion_matrices.items():
    cm_df = pd.DataFrame(
        cm,
        index=["Actual_Non_Referable", "Actual_Referable"],
        columns=["Predicted_Non_Referable", "Predicted_Referable"]
    )

    cm_df.to_csv(
        OUTPUT_DIR / f"{model_name}_binary_confusion_matrix.csv"
    )

print("Binary confusion matrices exported successfully.")

for model_name, cm in binary_confusion_matrices.items():
    print(f"\n{model_name}")
    print(cm)

Binary confusion matrices exported successfully.

ResNet50_best
[[134  17]
 [ 31 273]]

ResNet50_finetuned
[[134  17]
 [ 31 273]]

EfficientNetB0_finetuned
[[147   4]
 [ 45 259]]
